# Theoretical report and project comments

We gather all the theoretical comments and reflections associated with the RA. It is not an isolated text, it must be complemented with tables and results shown in the notebook, which is more extensive. Not all of them are shown here due to size, which a jup`yter notebook solves by allowing you to scroll over the displayed results. The raw and clean corpus, as well as the label dictionary, are generated as documents. The HASH of the clean and raw text is shown in the notebook at the end of the preprocessing, and the seed is the first thing shown in the same cell as the library imports.

---

## PART 1 – Corpus, preprocessing and representations

### TASK 1.1 – Corpus description and quality control

This task starts with a synthetic corpus of 90 urban messages in Spanish, balanced across five thematic categories and three urgency levels. The categories are: ResiduosLimpieza, MovilidadTransporte, AlumbradoPublico, RuidoNocturno and ZonasVerdes, with 18 messages per category. The urgency levels are High, Medium and Low, with 30 messages in each level.

The corpus is deliberately constructed in a controlled manner. It gathers prototypical incidents of a city (Valencia) with relatively standard language, although including urban abbreviations and colloquial variants. This facilitates the design of preprocessing rules while introducing some realistic variability in the textual surface.

The SHA-256 hash is calculated for both the “raw corpus” (concatenation of unprocessed texts) and the “clean corpus” (concatenation of texts after the cleaning and normalization pipeline), which allows verifying that transformation has indeed occurred and that content has not been accidentally lost or mixed up. During execution, the hashes differ, which confirms the controlled modification of the textual surface.

A summary sheet of the corpus (`fichacorpus`) is also generated where various data about it is documented.

---

### TASK 1.2 – Linguistic preprocessing and comments

The preprocessing pipeline is designed with several chained functions:

1. **Basic cleaning** (removal of HTML/URLs, conversion to lowercase, space normalization).
2. **Expansion of urban abbreviations** (for example `c.` → `calle`, `avd.` → `avenida`, `pza` → `plaza`) and correction of frequent errors (`xq` → `porque`, `haber si` → `a ver si`), to reduce lexical dispersion and bring the messages closer to a canonical form.
3. **Accent removal** via Unicode normalization (NFD) to prevent `Blasco Ibáñez` and `Blasco Ibanez` from being treated as different words.
4. **Tokenization and filtering** with spaCy, in which spaces, punctuation marks, and stopwords are excluded, but it is decided to retain some negations (`no`, `ni`, `nunca`, `tampoco`, `sin`) because they can alter the meaning and priority (for example, “no hay luz” versus “hay luz”). 
5. **Lemmatization** in Spanish, discarding empty lemmas, the special lemma `-PRON-`, and too short non-numeric tokens; additionally, stopwords are filtered again.
6. **Entity extraction** (people, organizations, places, etc.) with spaCy, which are saved for possible later use as context, although they are not yet incorporated as a direct feature in the classification models.

Finally, it is verified that the quality control does not remove messages in this case (before and after there are 90 messages), and it is checked that the hash of the clean corpus differs from the hash of the raw corpus, which validates the effectiveness of the pipeline without loss of documents.

---

### TASK 1.3 – BoW, TF‑IDF, and spaCy embeddings representations

In this task, three ways of representing the documents are compared:

1. **Bag of Words (BoW)** with `CountVectorizer`, which generates a count matrix of size (90, 772). Its interpretability is high, the computational cost is low, and the semantic capacity is limited: the representation captures presence and frequency, but not finer relationships of meaning.
2. **Global TF‑IDF**, which uses the same dimensionality (90, 772), but with real weights that reflect relative importance (frequency in a document versus global frequency in the corpus). Its interpretability remains high, its cost is low-medium, and the semantic capacity is at a medium level, useful for classification and retrieval.
3. **spaCy embeddings**, which generate dense vectors of dimension 96 per document. Their interpretability is medium-low, the cost is medium, and the semantic capacity is also medium, suitable for complementary semantic similarity and detection of proximities that do not depend only on lexical matches.

These differences are synthesized in a table that can be seen in the Jupyter notebook, which summarizes the dimensionality, interpretability, computational cost, and main utility of each representation.

In addition, the dominant TF‑IDF terms are calculated for each thematic category, which allows observing how certain words (`farola`, `contendedor`, `ruido`, `jardín`) strongly characterize each type of incident. This inspection serves as a qualitative justification that the vectorization is aligned with the conceptual structure of the corpus.

---

## PART 2 – Supervised models and topic modeling

### TASK 2.1 – Thematic classification and urgency classification

Two supervised problems are proposed in this task:

- Thematic category classification (`categoria`).
- Urgency level classification (`urgencia`).

Two pipelines are built with TF-IDF:
- **TFIDF NaiveBayes**: `TfidfVectorizer` + `MultinomialNB`.
- **TFIDF LogisticRegression**: `TfidfVectorizer` + `LogisticRegression`. 

Stratified cross-validation (`StratifiedKFold`) is used with 5 folds and accuracy, macro precision, macro recall, macro F1, and weighted F1 are measured. Weighted F1 is prioritized as the main metric because the practice warns of the sensitivity to differences between classes and the importance of weighting by support, even though the classes are already balanced.

The results obtained for the thematic classification show that:

- Logistic regression offers a better balance between performance and interpretability, with a higher weighted F1 than Naive Bayes, although the values remain modest (small corpus, high variety of formulations).
- The classification report by category highlights that some classes (RuidoNocturno) have high precision but low coverage (many false negatives), while others (MovilidadTransporte) show better recall. This is discussed as an example of a trade-off between sensitivity and specificity.

Errors are also collected in a "THEMATIC CLASSIFICATION ERRORS" dataframe, which illustrates plausible confusions: for example, ZonasVerdes messages that the model assigns to MovilidadTransporte when there is a reference to a bike lane, or RuidoNocturno messages confused with AlumbradoPublico if the description mixes light and noise.

For the urgency classification, the comparison shows similar results between Naive Bayes and logistic regression, with accuracies around 0.55 and a moderate weighted F1. It is emphasized that the boundaries between High, Medium, and Low are more conceptual than lexical: the model depends heavily on risk words ("fire", "danger", "hit-and-run", "blackout"), but it can doubt in ambiguous or more informative messages.

- **Logistic Regression** in both cases, for keeping the same model, maintaining equal metrics in the case of urgency and better ones in the case of theme.
- **TF-IDF** is maintained as vectorization due to its lexical traceability, low computational cost, and ease of auditing in a municipal context.

---

### TASK 2.2 – Topic modeling with LSI and LDA, and evaluation (variance vs. perplexity)

In this task, two approaches to topic modeling are compared:

- **LSI (Latent Semantic Indexing)** implemented using `TruncatedSVD` on a TF‑IDF matrix.
- **LDA (Latent Dirichlet Allocation)** implemented on a BoW matrix (`CountVectorizer`).


Two configurations of each model are trained (k=5 and k=7) and evaluated with the natural criteria of each approach:
- LSI is evaluated with cumulative explained variance to measure how much variation in the data is captured by the latent subspace of dimension k.
- LDA is evaluated with perplexity on the document matrix to estimate how well the probabilistic model predicts the data. A lower perplexity indicates a better statistical fit to the corpus.

- For LSI, the cumulative explained variance for k=5 and k=7 increases as k increases but remains moderate, possibly due to the size of the corpus. This is evidence that the model captures a significant fraction of the semantic structure but not all of it.
- For LDA, the perplexity for k=5 is better than for k=7, so the configuration for k=5 better fits the set of documents.

To continue with the task, LDA k=5 is adopted.

Then, the topics are listed with their most representative words. For LDA k=5, we could assign a name to some topics: Mobility to topic 4 and Maintenance to topic 1. The rest present quite a lot of noise that at first glance would not make us think what context they suggest, since among them they repeat words like streetlight or trash. A crossover (`tablacrucetopicos`) is constructed between the dominant topic and the real category, which allows us to see which topics align better with each urban category.

---

### TASK 2.3 – Semantic similarity, graph and incident retrieval

This task integrates several pieces:

1. A semantic retrieval model based on TF‑IDF and cosine similarity.
2. Classifiers trained to estimate category and urgency of new queries.
3. A semantic graph of lemma co-occurrences and centrality analysis (degree and betweenness).

A retrieval function is defined that:
- Preprocesses the citizen query with the same pipeline as the corpus.
- Projects it into the global TF‑IDF space.
- Calculates cosine similarity against all documents.
- Retrieves the most similar messages with their categories, urgencies, and original texts.

It is accompanied by a rule-based action recommendation function:

- If the predicted urgency is High → immediate referral to the municipal service and opening of a priority incident.
- If the urgency is Medium → ordinary registration with technical review in the same work cycle.
- If the urgency is Low → informative response or incorporation into planning.

Several example queries are tested and the results are discussed. For example:
- “There is broken glass and danger in a neighborhood playground” is classified as ZonasVerdes and High urgency, retrieves highly related incidents (broken glass in dog area, broken swings with protruding irons) and recommends immediate referral.
- “The streetlight on my street has been off since last night” is classified as AlumbradoPublico and High urgency, retrieving blackouts and streetlights with problems.
- “It smells terrible next to the dumpsters and there is trash outside” is classified as ResiduosLimpieza with Medium urgency and prioritizes similar incidents of odors and accumulated trash. 
- “There is a party with very loud music in an apartment in the early hours of the morning” is classified as RuidoNocturno, High urgency, and retrieves illegal parties and queries about the noise ordinance.

These tests illustrate that the system combines classification, retrieval, and rules, building a simple but traceable “municipal assistant” type behavior.

In parallel, a lexical co-occurrence graph is built (nodes = lemmas with length≥3, edges weighted by the number of co-occurrences in messages), and degree and betweenness centralities are calculated. It is observed that terms like `calle`, `avenida`, `farola`, `pasar`, `plaza`, `hacer`, `roto`, `poder`, `parque`, `contenedor` emerge as central nodes.

This analysis is evidence that, from a semantic and urban point of view:

- `calle` acts as a structural core connecting multiple types of incidents (waste, lighting, mobility).
- `farola`, `ruido`, `arbol`, and `ayuntamiento` appear as terms that articulate relevant thematic subnetworks (`farola` between lighting and security, `ruido` between leisure and ordinance, `arbol` between green areas and security).

---

## PART 3 – Conversational architecture and design decisions

### 3.1 Architecture of the conversational component (NLU, Dialog Manager, NLG)

The prototype is explicitly designed as a **task-oriented dialogue system** (not as an open chat), because the objective is clearly defined: to receive citizen incidents and queries, analyze them, retrieve related information, offer an initial response, and route critical cases to the appropriate municipal service.

The architecture adopts the following scheme:

1. **NLU (Natural Language Understanding)**  
   - Receives the citizen's message in free text.
   - Applies the preprocessing pipeline: cleaning, normalization, tokenization, lemmatization, entity extraction.
   - Uses the trained classifiers (theme and urgency) to estimate the basic intent of the message and its operational priority.
   - This separation allows the comprehension part to remain independent from the rest of the system, facilitating its future improvement (for example, replacing models if the corpus changes without touching manager rules).

2. **Dialogue Manager**  
   - Acts as the decision core of the system.
   - Based on category, urgency, relevant entities, and results from the retrieval module (similar messages), it decides whether it should:
     - Respond informatively.
     - Register the incident without a special alert.
     - Activate a critical alert escalation.

3. **NLG (Natural Language Generation)**  
   - Transforms the system's decision into:
     - An initial response for the citizen (for example, confirmation of receipt, explanation of the category and urgency level, guidance on next steps).
     - A structured internal escalation for the corresponding municipal service (for example, estimated category and urgency, similar messages, detected risk vocabulary). 
   - In this prototype, generation is implemented in a controlled and traceable manner, using templates and rules, not through free generation with large models like LLMs

**The diagram is found at the end of the Jupyter notebook**

#### Decision on the generative component (LLM vs. controlled simulation)

In this prototype, a **controlled simulation** of the conversational component is chosen, without integrating an LLM. Among the reasons for this decision:

1. Reproducibility: avoids depending on external services that can change their APIs, models, or access policies; allows anyone with the notebook and standard libraries to reproduce all outputs.

2. Privacy and data protection: in a real municipal context, sending citizen messages to external services can raise data protection and confidentiality issues; local simulation reduces the risk of exposing sensitive information and is more in line with prudent public administration practices.

3. Auditability and traceability: each generated response can be related to rules, models, and results visible in the notebook; furthermore, this facilitates review by municipal technicians and academic evaluators, who can see exactly how a decision was reached.

4. Technical and economic cost: the integration of LLMs (especially paid ones) may be unnecessary in a first phase and complicate deployment in a teaching environment.

The main recognized limitation is that the generated responses are less flexible and natural than those of a real LLM. However, for a limited context in which the use of the prototype is limited, it is sufficient, since only a first contact is sought. The agent would not solve any problem by itself, it only serves as a repository and cooperative manager, providing guidelines and preliminary responses.


#### Critical alert derivation rules

In the prototype, a scheme is implemented in which a critical alert is triggered when at least some of the following conditions occur:

- Estimated urgency High.
- Presence of risk vocabulary (`fire`, `power outage`, `run over`, `sparks`, `fallen branch`, etc.).
- Sensitive urban entities (central squares, busy streets, key infrastructures).
- High similarity with messages previously labeled as critical in the corpus.

These rules allow the system to go beyond classification, becoming an operational triage tool that helps prioritize potentially serious cases. However, it is emphasized that the generated alerts are decision aids, not definitive administrative decisions, and that human intervention is still necessary, especially in contexts of high responsibility (emergencies, citizen security).

---

### 3.2 Evaluation of the prompts

The eight test queries, belonging to the initial corpus for this part, include easy, ambiguous, and critical cases, which allows observing the behavior of the system in heterogeneous scenarios consistent with the selected urban case. The comparison between the two versions of the classification prompt shows that the second version is superior, which is expected since it has a higher degree of specificity: the use of closed categories, explicit delimiters, restrictions against invention, and a fixed JSON output improves coherence, stability, and ease of auditing.

Overall, the designed prompts allow integrating the conversational layer with the modules previously developed in the notebook. However, their final reliability still depends on the quality of the corpus, the consistency of the classification, and human supervision in critical or ambiguous cases.

For the compared prompts, we make the following comparative table: 

| Version | Role/context                                     | Constraints                                                          | Output format                                                        | Categories/priorities |
| ------- | ------------------------------------------------ | -------------------------------------------------------------------- | -------------------------------------------------------------------- | ---------------------- |
| v1      | Poorly defined                                   | Almost none                                                          | Generic JSON                                                         | Not enumerated         |
| v2      | Municipal analyst role, clear urban context      | Explicit (do not invent data, do not add rules, handle ambiguity)    | JSON with fixed keys (category, priority, summary, justification)    | Listed in the prompt   |

---

### 3.3 Comprehensive validation of the prototype

The comprehensive validation has been carried out on eight queries covering urgent incidents, informative queries, and ambiguous cases. The objective has been to evaluate the end-to-end behavior of the prototype: preprocessing, thematic classification, urgency estimation, retrieval of similar incidents, and action recommendation.

Errors appear mainly in queries with lexical overlap between categories. The case of “A badly parked car blocks the bus's path” well illustrates this limitation: the system tends to approximate it to `Residuos_Limpieza` examples due to the coincidence with expressions like “blocks the path”, when conceptually the case belongs to `Movilidad_Transporte`.

The average latency per query is low and compatible with an academic prototype or a municipal proof of concept. However, this figure must be interpreted with caution, as it has been measured on a small corpus, resident in memory, and without calls to external generation services.

The results are collected in two tables, one about the 8 queries and another about those with an error or mismatch, which can be found in the notebook. Other measured aspects such as latency are also collected here:

GLOBAL INDICATORS
- Accuracy in category: 0.8750
- Accuracy in urgency: 0.5000
- Mean latency per query: 20.59 ms
- Latency standard deviation: 3.34 ms

#### Limits and risks of the conversational system

The system designed in this project presents several limits:

- The corpus is relatively small and controlled, which reduces its lexical coverage and may affect its performance when faced with more heterogeneous or ambiguous messages. (it is understood that this is because an academic illustration is sought)
- TF-IDF based retrieval relies heavily on lexical overlap: very different reformulations can result in being less similar even if they describe equivalent problems.
- Critical alert rules can produce false positives or false negatives if the message uses unanticipated expressions or if the semantic prioritization fails to capture important nuances (for example, irony, sarcasm, or technical details). 

Given these limitations, any real deployment would require: constant human supervision; expansion and diversification of the corpus (inclusion of real data, dialectal variants, audio and transcriptions); performance monitoring (health metrics and periodic auditing); possible progressive integration of more advanced models (including LLMs, BERT or their variants) under clear policies of transparency and data protection.

---

### 3.4 Governance, ethics, and professional communication

#### Privacy and data protection

The corpus used in this practical exercise is synthetic and has been built for teaching purposes, so it does not contain real personal data. Even so, a real municipal deployment would require specific measures for anonymization, data minimization, access control, and usage traceability. This requirement is especially important in a system that receives free text, since citizens can include exact addresses, proper names, phone numbers, license plates, or other sensitive references. Therefore, before storing or reusing the queries, it would be necessary to apply automatic mechanisms for detecting and pseudonymizing personal data. This falls within the framework of the GDPR where it is requested, for example, that the user can access their data, has the right to be forgotten (that the system deletes their data), and that the system only collects the necessary data for each purpose if appropriate in more advanced phases. Specifically, the system would require a specific anonymization layer prior to the NLP module, combining NER and rules to detect sensitive private data (names, addresses, phone numbers, license plates) and replacing it using systematic pseudonyms (e.g. STREET_ADDRESS_1, PERSON_1). This layer should be integrated with data minimization policies, access control, and limited log retention, so that only already anonymized texts are stored along with category, urgency, and destination service tags

#### Human supervision and responsibility

The developed prototype must be understood as a support tool for the initial triage of which situations in the municipality are urgent and what situation each one is in, not as a substitute for administrative decision-making. Its reasonable role is to help classify, prioritize, and retrieve similar incidents, but the final decision in critical cases should remain under human supervision. This principle is especially relevant in messages classified as `High` urgency or in ambiguous queries. If the system assigns a lower priority to a truly serious case, the operational and social impact can be significant. Therefore, automation should be considered as intelligent assistance and not as autonomous case resolution.

#### Auditing and traceability

For the system to be auditable, every output should be reproducible. Included in this traceability are best practices such as using reproducible seeds, computing hashes of the corpus, and explicitly separating classification, retrieval, and action rules. In a real-world environment, this traceability should be extended with model versioning, logging of triggered rules, retaining retrieved incidents as context, and storing response templates. Specifically, each interaction must record the system and user prompts, model version, inference parameters, and retrieved sources, listing the incidents along with their category, urgency, and TF-IDF similarity to others. Additionally, the system must maintain error tables—like those we presented at the end of the comprehensive model validation in the notebook—to monitor discrepancies in category and urgency, and to feed into prompt refactoring or model fine-tuning processes. This explainability is crucial in a public system, both for technical debugging and for accountability.

### Recommendations for the city council

The recommendations for the city council before implementing the prototype can be summarized as follows: the viability of a scoped pilot as a semi-automated triage system with the aforementioned measures, especially the expansion of the corpus; and the deployment condition of including robust anonymization, human oversight in critical decisions, and exhaustive traceability logging. The key limitations are the reliance on a controlled corpus, the sensitivity to ambiguous messages, and the inability to replace professional judgment in sensitive decisions, meaning the system must be clearly positioned as a technical support tool.